# Read Tables

In [0]:
customers_df = spark.table("workspace.default.bronze_customers")
orders_df = spark.table("workspace.default.bronze_orders")
items_df = spark.table("workspace.default.bronze_items")
payments_df = spark.table("workspace.default.bronze_payments")
products_df = spark.table("workspace.default.bronze_products")

In [0]:
print("Customers :", customers_df.count())
print("Orders    :", orders_df.count())
print("Items     :", items_df.count())
print("Payments  :", payments_df.count())
print("Products  :", products_df.count())

Customers : 15000
Orders    : 50000
Items     : 86328
Payments  : 57388
Products  : 3000


In [0]:
customers_df.printSchema()
orders_df.printSchema()
items_df.printSchema()
payments_df.printSchema()
products_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |

In [0]:
from pyspark.sql.functions import col, sum, when

orders_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in orders_df.columns
]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|                0|                           0|                        25034|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



###Convert Date Columns

In [0]:
from pyspark.sql.functions import to_timestamp

orders_df = (
    orders_df
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp")
    )
    .withColumn(
        "order_approved_at",
        to_timestamp("order_approved_at")
    )
    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp("order_delivered_carrier_date")
    )
    .withColumn(
        "order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date")
    )
    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date")
    )
)

In [0]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



### Create delivery_days

In [0]:
from pyspark.sql.functions import datediff

orders_df = orders_df.withColumn(
    "delivery_days",
    datediff(
        "order_delivered_customer_date",
        "order_purchase_timestamp"
    )
)

In [0]:

orders_df.select(
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_days"
).show(10, truncate=False)

+------------------------+-----------------------------+-------------+
|order_purchase_timestamp|order_delivered_customer_date|delivery_days|
+------------------------+-----------------------------+-------------+
|2023-06-15 14:30:00     |2023-06-19 14:30:00          |4            |
|2021-06-12 11:02:00     |NULL                         |NULL         |
|2022-03-31 19:38:00     |2022-04-14 19:38:00          |14           |
|2021-09-09 22:29:00     |2021-09-23 22:29:00          |14           |
|2022-08-27 07:46:00     |NULL                         |NULL         |
|2023-05-26 16:27:00     |NULL                         |NULL         |
|2021-09-21 15:50:00     |NULL                         |NULL         |
|2021-01-19 14:24:00     |2021-01-28 14:24:00          |9            |
|2021-07-07 23:18:00     |2021-07-20 23:18:00          |13           |
|2021-02-21 04:01:00     |NULL                         |NULL         |
+------------------------+-----------------------------+-------------+
only s

In [0]:
orders_df.filter(
    orders_df.delivery_days < 0
).count()

0

### Check Duplicates

In [0]:
print("Customers Before :", customers_df.count())

customers_df = customers_df.dropDuplicates()

print("Customers After  :", customers_df.count())

Customers Before : 15000
Customers After  : 15000


In [0]:
print("Orders Before :", orders_df.count())

orders_df = orders_df.dropDuplicates()

print("Orders After  :", orders_df.count())

Orders Before : 50000
Orders After  : 50000


In [0]:
print("Items Before :", items_df.count())

items_df = items_df.dropDuplicates()

print("Items After  :", items_df.count())

Items Before : 86328
Items After  : 86328


In [0]:
print("Payments Before :", payments_df.count())

payments_df = payments_df.dropDuplicates()

print("Payments After  :", payments_df.count())

Payments Before : 57388
Payments After  : 57388


In [0]:
print("Products Before :", products_df.count())

products_df = products_df.dropDuplicates()

print("Products After  :", products_df.count())

Products Before : 3000
Products After  : 3000


###Standardize Text Columns

In [0]:
from pyspark.sql.functions import lower, trim

orders_df = orders_df.withColumn(
    "order_status",
    lower(trim("order_status"))
)

In [0]:
orders_df.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|   delivered|
|   cancelled|
|     shipped|
|    invoiced|
|  processing|
+------------+



###Business Rule Validation

In [0]:
from pyspark.sql.functions import col

items_df.filter(col("price") < 0).count()

0

In [0]:
items_df.filter(col("freight_value") < 0).count()

0

In [0]:
payments_df.filter(col("payment_value") <= 0).count()

0

##Save Silver Tables

In [0]:
customers_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.silver_customers")

orders_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.silver_orders")

items_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.silver_items")

payments_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.silver_payments")

products_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.default.silver_products")

In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_items,false
default,bronze_orders,false
default,bronze_payments,false
default,bronze_products,false
default,silver_customers,false
default,silver_items,false
default,silver_orders,false
default,silver_payments,false
default,silver_products,false
